# AutoML-M01: Baselines

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Modelos baseline como referencia. Se ejecutan para **Caso D** y **Caso D_strict**.

## 📊 Modelos: DummyClassifier, LogReg, RandomForest, GradientBoosting, XGBoost

## 📝 Nota
Datos auditados (F3-M08). Sin leakage, sin constantes, sin traidoras.

In [6]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys, warnings, time
from pathlib import Path
warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️ XGBoost no disponible')

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es
RANDOM_STATE = 42
FRAMEWORK = 'baselines'

# DATASETS: Caso D y D_strict (limpios, sin leakage)
DATASETS = {
    'D': RUTA_AUTOML / 'df_exp_automl_D.parquet',
    'D_strict': RUTA_AUTOML / 'dataset_final_tfm.parquet',
}

info_entorno()

# Verificación anti-leakage
for caso, ruta in DATASETS.items():
    df_tmp = pd.read_parquet(ruta)
    n_cols = df_tmp.shape[1]
    del df_tmp
    print(f'  ✅ Caso {caso}: {ruta.name} ({n_cols} cols) — LIMPIO')
print('✅ Verificación anti-leakage superada')

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: ENTRENAR MODELOS (CASO A + CASO B)
# ============================================================================

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te, framework, caso):
    pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', modelo)])
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    try:
        y_prob = pipe.predict_proba(X_te)[:, 1]
    except:
        y_prob = np.zeros(len(y_te))
    t_total = time.time() - t0
    return {
        'caso': caso, 'framework': framework, 'model_name': nombre,
        'accuracy': accuracy_score(y_te, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall': recall_score(y_te, y_pred, zero_division=0),
        'f1': f1_score(y_te, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_te, y_prob) if y_prob.sum() > 0 else 0,
        'mcc': matthews_corrcoef(y_te, y_pred),
        'kappa': cohen_kappa_score(y_te, y_pred),
        'log_loss': log_loss(y_te, y_prob) if y_prob.sum() > 0 else 999,
        'train_time_sec': round(t_total, 2)
    }

all_results = []

for caso, ruta in DATASETS.items():
    print(f'\n{"="*70}')
    print(f'CASO {caso}: {ruta.name}')
    print(f'{"="*70}')

    df = pd.read_parquet(ruta)
    TARGET = 'abandono'
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    print(f'Dataset: {len(df):,} × {len(df.columns)} cols ({len(X.columns)} features)')
    print(f'Target: {(y==1).sum():,} abandono ({(y==1).mean()*100:.1f}%)')

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

    modelos = [
        ('DummyClassifier_stratified', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
        ('DummyClassifier_majority', DummyClassifier(strategy='most_frequent')),
        ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)),
        ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
        ('GradientBoosting', GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
    ]
    if HAS_XGB:
        modelos.append(('XGBoost', XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss', verbosity=0)))

    for i, (nombre, modelo) in enumerate(modelos, 1):
        print(f'  [{i}/{len(modelos)}] {nombre}...', end=' ')
        m = evaluar_modelo(nombre, modelo, X_train, y_train, X_test, y_test, FRAMEWORK, caso)
        print(f'AUC={m["auc_roc"]:.4f} | F1={m["f1"]:.4f} | MCC={m["mcc"]:.4f} | {m["train_time_sec"]}s')
        all_results.append(m)

df_resultados = pd.DataFrame(all_results)
df_resultados = df_resultados.sort_values(['caso', 'f1'], ascending=[True, False]).reset_index(drop=True)

for caso in DATASETS.keys():
    print(f'\n--- RANKING CASO {caso} (por F1) ---')
    mask = df_resultados['caso'] == caso
    print(df_resultados.loc[mask, ['model_name', 'accuracy', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].to_string(index=False))


CASO D: df_exp_automl_D.parquet
Dataset: 33,621 × 22 cols (21 features)
Target: 9,833 abandono (29.2%)
AUC=0.4908 | F1=0.2738 | MCC=-0.0186 | 0.11s
  [2/6] DummyClassifier_majority... AUC=0.0000 | F1=0.0000 | MCC=0.0000 | 0.09s
AUC=0.9002 | F1=0.7333 | MCC=0.6334 | 2.48s
AUC=0.9417 | F1=0.7927 | MCC=0.7200 | 1.34s
AUC=0.9354 | F1=0.7733 | MCC=0.6924 | 7.48s
AUC=0.9433 | F1=0.8002 | MCC=0.7254 | 0.75s

CASO D_strict: dataset_final_tfm.parquet
Dataset: 33,621 × 20 cols (19 features)
Target: 9,833 abandono (29.2%)
AUC=0.4908 | F1=0.2738 | MCC=-0.0186 | 0.1s
  [2/6] DummyClassifier_majority... AUC=0.0000 | F1=0.0000 | MCC=0.0000 | 0.1s
AUC=0.8883 | F1=0.7213 | MCC=0.6187 | 2.57s
AUC=0.9273 | F1=0.7782 | MCC=0.7019 | 1.36s
AUC=0.9202 | F1=0.7589 | MCC=0.6756 | 5.96s
AUC=0.9288 | F1=0.7821 | MCC=0.7022 | 1.29s

--- RANKING CASO D (por F1) ---
                model_name  accuracy       f1  auc_roc       mcc  train_time_sec
                   XGBoost  0.888867 0.800214 0.943308  0.725366     

In [3]:
# ============================================================================
# CELDA 3: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_baselines.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} filas (caso D + D_strict)')

💾 results_baselines.parquet: 12 filas (caso D + D_strict)


In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS Y HTML
# ============================================================================

graficos_b64 = {}
nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase_automl', modulo_activo='m01')

# Gráfico: D vs D_strict barras agrupadas (mejores modelos)
for caso in DATASETS.keys():
    mask = (df_resultados['caso'] == caso) & (~df_resultados['model_name'].str.startswith('Dummy'))
    df_plot = df_resultados[mask].copy()
    metricas_plot = ['accuracy', 'f1', 'auc_roc', 'mcc', 'balanced_accuracy']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(df_plot))
    width = 0.15
    colores = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']
    for j, (met, col) in enumerate(zip(metricas_plot, colores)):
        ax.bar(x + j*width, df_plot[met], width, label=met, color=col)
    ax.set_xticks(x + width*2)
    ax.set_xticklabels(df_plot['model_name'], rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'Baselines Caso {caso}: Métricas', fontweight='bold', fontsize=14)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    graficos_b64[f'barras_{caso}'] = figura_a_base64(fig)
    plt.close()

# HTML
contenido_html = ''

for caso in DATASETS.keys():
    mask = df_resultados['caso'] == caso
    df_c = df_resultados[mask]
    mejor = df_c.iloc[0]

    tabla = '<table style="width:100%;border-collapse:collapse;">\n'
    tabla += '<tr style="background:#3182ce;color:white;">'
    for col in ['Modelo', 'Acc', 'Bal.Acc', 'Prec', 'Recall', 'F1', 'AUC', 'MCC', 'Kappa', 'LogLoss', 'Tiempo']:
        tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
    tabla += '</tr>\n'
    for i, row in df_c.iterrows():
        bg = '#f7fafc' if i % 2 == 1 else 'white'
        tabla += f'<tr style="background:{bg};">'
        tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
        for c in ['accuracy','balanced_accuracy','precision','recall','f1','auc_roc','mcc','kappa']:
            v = row[c]; color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
            tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["log_loss"]:.4f}</td>'
        tabla += f'<td style="text-align:center;font-size:11px;">{row["train_time_sec"]:.1f}s</td></tr>\n'
    tabla += '</table>'

    graf = f'<img src="data:image/png;base64,{graficos_b64[f"barras_{caso}"]}" style="max-width:100%;border-radius:8px;">'

    desc_caso = 'Alerta temprana (21 features)' if caso == 'D' else 'Producción (19 features)'
    n_feat = len(df_c.iloc[0]) - 4  # approximate
    contenido_html += generar_seccion_html(
        f'📊 Caso {caso}: {desc_caso}',
        f'<p><strong>Mejor:</strong> {mejor["model_name"]} (F1={mejor["f1"]:.4f}, AUC={mejor["auc_roc"]:.4f})</p>\n{tabla}\n{graf}')

# KPIs del mejor global
casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]
KPIS = [
    {'valor': str(len(df_resultados)//2), 'titulo': 'Modelos'},
    {'valor': f'{mejor_d["f1"]:.4f}', 'titulo': 'Mejor F1 (D)'},
    {'valor': f'{mejor_ds["f1"]:.4f}', 'titulo': 'Mejor F1 (D_strict)'},
    {'valor': '2', 'titulo': 'Casos'},
]

html = render_base_html(
    titulo='M01: Baselines', icono='📊', subtitulo='Pre-Modelado AutoML (Caso D + D_strict)',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + contenido_html,
    notebook_nombre='fautoml_m01_baselines.ipynb', notebook_carpeta='fase_automl')
ruta_html = RUTA_FASE_AUTOML_HTML / 'm01_baselines.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m01_baselines.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m01_baselines.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN
# ============================================================================

casos_k = list(DATASETS.keys())
mejor_d = df_resultados[df_resultados['caso']==casos_k[0]].iloc[0]
mejor_ds = df_resultados[df_resultados['caso']==casos_k[1]].iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M01 COMPLETADO')
print('='*60)
print(f'Modelos por caso: {len(df_resultados)//2}')
print(f'Caso D mejor: {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f}, AUC={mejor_d["auc_roc"]:.4f})')
print(f'Caso D_strict mejor: {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f}, AUC={mejor_ds["auc_roc"]:.4f})')
print(f'Resultados: {ruta_resultados}')
print(f'HTML: {ruta_html}')
print(f'\n🎯 Siguiente: fautoml_m02_lazypredict.ipynb')
print('='*60)


✅ AUTOML-M01 COMPLETADO
Modelos por caso: 6
Caso D mejor: XGBoost (F1=0.8002, AUC=0.9433)
Caso D_strict mejor: XGBoost (F1=0.7821, AUC=0.9288)
Resultados: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_baselines.parquet
HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m01_baselines.html

🎯 Siguiente: fautoml_m02_lazypredict.ipynb
